# AHR-only verified learning and continuation games

Purpose: learn explicit verified tactical obligations, demonstrate heldout competence and stronger offline teaching, then collect fresh outcomes through cheap continuation games and a minority of empty-board games. GNU Go is an opponent and never supplies policy labels. Old weak player choices never supervise the policy.

Inputs: version-five model/search definitions, hash-verified train/development tactical curriculum, pinned GNU binary. Outputs: real optimizer checkpoints, immutable train-only game archives, progress and gate reports. Expected full experiment runtime: the pipeline assigns approximately 16 hours of training within an 18-hour GPU-backed experiment. Timings distinguish active time, optimization and search; this is not a claim of continuously busy CUDA kernels.

Checkpoint: `checkpoints/<run_id>/`, full model/Adam/scheduler/scaler, Python/NumPy/Torch/private GPU-1 RNG, sampler, replay/reanalysis, historical actors and pending move. Latest two valid resumable checkpoints plus best are retained. One writer only; each phase boundary and at most 32 events/250 updates/10 minutes is saved atomically.

Exact resume: load exports from notebooks 00–04, then `v5_run_learning(project, run_id, saved_config, run_until_games=cumulative_target, deadline_unix=new_segment_deadline)`. Configuration and source identity must match; invalid manifests never silently restart. Run All checks definitions and displays existing progress without launching training. The controller starts the already-authorized long run in detached tmux after recovery validation.

In [ ]:
from pathlib import Path
import nbformat,json
project=Path.cwd().resolve()
while not (project/'notebooks/00_instance_setup_and_recovery.ipynb').exists():project=project.parent
for cell in nbformat.read(project/'experiments/ahr_native_v5/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(cell.source,globals())
v5_load_v4(project,globals())
for name in ('01_model_and_search.ipynb','02_verified_curriculum.ipynb'):
    v5_import_exports(project/V5_ROOT/name,globals())


In [ ]:
from dataclasses import dataclass, asdict, field
from collections import Counter, defaultdict, OrderedDict
import copy, hashlib, json, math, os, random, subprocess, sys, time, uuid
import numpy as np
import torch

V5_TRAIN_SCHEMA='ahr_native_competitive_training_v5'
V5_TRAJECTORY_SCHEMA='ahr_native_training_trajectory_v5'
V5_TRAIN_NOTEBOOK='experiments/ahr_native_v5/03_learning.ipynb'
V5_PRIOR_RUNS=()

@dataclass(frozen=True)
class V5TrainConfig:
    seed:int=2026100201
    scheduled_games:int=4096
    warmup_updates:int=256
    updates_per_game:int=16
    batch_size:int=64
    learning_rate:float=0.0002
    weight_decay:float=0.0001
    device:str='cuda:1'
    board_size:int=9
    komi:float=7.5
    max_plies:int=256
    search:dict=field(default_factory=lambda:{'time_budget_s':.25,'max_depth':2})
    reanalysis_search:dict=field(default_factory=lambda:{'time_budget_s':3.0,'max_depth':4,'max_nodes':2048,'branch_width':12})
    reanalysis_per_game:int=8
    tactical_fraction:float=0.5
    reanalysis_fraction:float=0.25
    replay_capacity_games:int=4096
    opponent_snapshots:int=8
    snapshot_every_games:int=16
    exploration_start:float=0.10
    exploration_end:float=0.02
    temperature_start:float=1.0
    temperature_end:float=0.25
    temperature_opening_plies:int=20
    exploration_decay_games:int=512
    gnu_level:int=1
    gnu_budget_s:float=5.0
    gnu_command_timeout_s:float=3.0
    prior_runs:tuple=V5_PRIOR_RUNS
    checkpoint_every_events:int=32
    checkpoint_every_optimizer_steps:int=250
    checkpoint_interval_s:float=600.0
    purpose:str='go_training'
    validation_scripted:bool=False
    validation_initial_game:int=0
    adaptive_warmup:bool=True
    gate_every_updates:int=256
    skill_capture_threshold:float=.99
    skill_overall_threshold:float=.95
    teacher_gate_needed:bool=True
    restart_fraction:float=.7
    validation_skip_learning_gate:bool=False
    validation_use_legacy_curriculum:bool=False
    def __post_init__(self):
        object.__setattr__(self,'prior_runs',tuple(self.prior_runs))
        for name in ('scheduled_games','updates_per_game','batch_size','max_plies','replay_capacity_games','opponent_snapshots','snapshot_every_games','exploration_decay_games','checkpoint_every_events','checkpoint_every_optimizer_steps'):
            if type(getattr(self,name)) is not int or getattr(self,name)<1:raise ValueError(name+' must be positive')
        if self.warmup_updates<0 or self.reanalysis_per_game<0:raise ValueError('Negative optional event count')
        if self.board_size!=9 or self.komi!=7.5:raise ValueError('This experiment fixes 9x9 and komi 7.5')
        if self.device not in ('cpu','cuda:1'):raise ValueError('Only CPU or project CUDA device 1 is authorized')
        if self.device=='cuda:1' and (not torch.cuda.is_available() or torch.cuda.device_count()<2):raise RuntimeError('CUDA device 1 unavailable')
        if not 0<=self.tactical_fraction<1 or not 0<=self.reanalysis_fraction<1 or self.tactical_fraction+self.reanalysis_fraction>=1:raise ValueError('Invalid batch mixture')
        if not 0<=self.exploration_end<=self.exploration_start<=1:raise ValueError('Invalid exploration schedule')
        if not 0<self.temperature_end<=self.temperature_start:raise ValueError('Invalid temperature schedule')
        if self.checkpoint_interval_s>600 or self.checkpoint_every_optimizer_steps>250 or self.checkpoint_every_events>32:raise ValueError('Checkpoint cadence too sparse')
        if self.gnu_level!=1 or self.gnu_command_timeout_s<=0 or self.gnu_budget_s<self.gnu_command_timeout_s:raise ValueError('Invalid GNU opponent budget')
        if self.purpose not in ('go_training','synthetic_smoke'):raise ValueError('Unknown purpose')
        if (self.validation_scripted or self.validation_initial_game) and self.purpose!='synthetic_smoke':raise ValueError('Fixtures must never masquerade as real training')
        if any(r not in V5_PRIOR_RUNS for r in self.prior_runs):raise ValueError('Only frozen prior B training runs may be imported')
        if (self.validation_skip_learning_gate or self.validation_use_legacy_curriculum) and self.purpose!='synthetic_smoke':raise ValueError('Only named fixtures may skip learning gate')
        if not 0<=self.restart_fraction<=1 or self.gate_every_updates<1:raise ValueError('Invalid restart/gate configuration')
        if not 0<self.skill_capture_threshold<=1 or not 0<self.skill_overall_threshold<=1:raise ValueError('Invalid actual competence thresholds')
        V5SearchConfig(**self.search);V5SearchConfig(**self.reanalysis_search)

def v5_training_path(project,relative):
    project=Path(project).resolve();path=(project/relative).resolve()
    if not path.is_relative_to(project):raise ValueError('Path escapes project')
    return path

def v5_training_prior(project,config):
    entries=[];manifests={}
    for run_id in config.prior_runs:
        root=Path(project)/'experiments/ahr_jepa_v3/data/trajectories'/run_id
        manifest_path=root/'manifest.json';manifest=json.loads(manifest_path.read_text())
        if manifest.get('run_id')!=run_id:raise RuntimeError('Prior archive identity changed')
        manifests[str(manifest_path.relative_to(project))]=file_sha256(manifest_path)
        training_report=json.loads((Path(project)/'reports/ahr_jepa_v3'/(run_id+'_training.json')).read_text())
        if training_report.get('status')!='COMPLETE' or training_report.get('config',{}).get('purpose')!='go_training':raise RuntimeError('Prior source is not completed real training')
        for entry in manifest['entries']:
            if entry.get('split')!='train' or entry.get('status') not in ('VERIFIED_TERMINAL','UNKNOWN'):raise RuntimeError('Nontraining or unfinished prior record')
            path=root/entry['file']
            if Path(entry['file']).name!=entry['file'] or file_sha256(path)!=entry['sha256']:raise RuntimeError('Corrupt prior trajectory')
            entries.append({**copy.deepcopy(entry),'path':str(path.relative_to(project)),'source_run':run_id,'origin':'prior_training','priority':1.0})
    return entries,{'manifests':manifests,'entries_digest':v5_digest(entries),'games':len(entries),'policy_labels':'discarded; terminal targets only until own AHR reanalysis'}

def v5_training_identity(project,config,curriculum,prior):
    return {'schema':V5_TRAIN_SCHEMA,'sources':v5_source_identity(project),'config':asdict(config),
            'curriculum':v5_digest({'train':curriculum['train'],'provenance':curriculum['provenance']}),
            'prior_training':prior,'gnu_binary':gnugo10_binary_identity(project)}

def v5_training_rng(device):
    cuda={}
    if str(device).startswith('cuda:'):cuda[str(torch.device(device).index)]=torch.cuda.get_rng_state(torch.device(device)).cpu()
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch_cpu':torch.get_rng_state(),'torch_cuda_used_devices':cuda}

def v5_restore_training_rng(rng):
    random.setstate(rng['python']);np.random.set_state(rng['numpy']);torch.set_rng_state(rng['torch_cpu'])
    for index,state in rng['torch_cuda_used_devices'].items():torch.cuda.set_rng_state(state,device=int(index))

def v5_read_training_checkpoint(run_dir,identity=None,best=False):
    run_dir=Path(run_dir);manifest_path=run_dir/'manifest.json'
    if not manifest_path.exists():
        if list(run_dir.glob('*.pt')):raise RuntimeError('Checkpoint payload exists without manifest')
        return None,None
    manifest=json.loads(manifest_path.read_text());entries=manifest.get('entries',[])
    if not entries or len({e['file'] for e in entries})!=len(entries):raise RuntimeError('Invalid checkpoint manifest')
    if manifest.get('latest')!=max(entries,key=lambda e:e['global_step'])['file']:raise RuntimeError('Latest checkpoint cursor mismatch')
    if manifest.get('best') not in {e['file'] for e in entries}:raise RuntimeError('Best checkpoint missing')
    loaded={}
    for entry in entries:
        state=_checkpoint_entry_state(run_dir,entry)
        if state is None or state.get('schema')!=V5_TRAIN_SCHEMA:raise RuntimeError('Corrupt or wrong-schema retained checkpoint')
        if identity is not None and state['identity']!=identity:raise RuntimeError('Frozen source, configuration, data, or GNU identity changed; refusing silent restart')
        if state['optimizer_step']<0 or state['global_step']<state['optimizer_step']:raise RuntimeError('Invalid checkpoint step')
        loaded[entry['file']]=(state,entry)
    return loaded[manifest['best' if best else 'latest']]


def v5_action_transform(action,symmetry,size=9):
    if action==size*size:return action
    row,col=divmod(int(action),size)
    if symmetry>=4:col=size-1-col
    for _ in range(symmetry%4):row,col=col,size-1-row
    return row*size+col

def v5_vector_transform(values,symmetry,size=9):
    result=[0.0]*len(values)
    for action,value in enumerate(values):result[v5_action_transform(action,symmetry,size)]=value
    return result

def v5_training_augment(example,symmetry):
    row=copy.deepcopy(example);board=from_dict(row['state']);row['state']=to_dict(transform_go(board,symmetry))
    for key in ('policy_target','tactical_policy_target','ownership_target'):
        if key in row and row[key] is not None:row[key]=v5_vector_transform(row[key],symmetry,board.size)
    if 'tactical_targets' in row:row['tactical_targets']=[v5_action_transform(a,symmetry,board.size) for a in row['tactical_targets']]
    for key in ('tactical_success_actions','tactical_failure_actions','tactical_unknown_actions'):
        if key in row:row[key]=[v5_action_transform(a,symmetry,board.size) for a in row[key]]
    row.pop('verification',None)
    row['symmetry']=symmetry
    return row

def v5_training_archive_root(project,run_id):
    if not isinstance(run_id,str) or Path(run_id).name!=run_id or run_id in ('.','..'):raise ValueError('Simple run identifier required')
    return Path(project)/'experiments/ahr_native_v5/data/trajectories'/run_id

def v5_verify_training_game(game):
    if game.get('schema')!=V5_TRAJECTORY_SCHEMA or game.get('split')!='train':raise ValueError('Wrong training trajectory schema or split')
    board=from_dict(game['initial_state'])
    if not game['records'] and game.get('status')!='UNKNOWN':raise ValueError('Empty terminal training trajectory')
    for record in game['records']:
        if record['state']!=to_dict(board):raise RuntimeError('Training chronology mismatch')
        if record.get('policy_weight',0)>0:selfplay09_policy(board,record['policy_target'])
        if record['actor']=='gnu' and record.get('policy_weight',0)!=0:raise RuntimeError('GNU policy imitation is forbidden')
        action=record['action'];child=play(board,action)
        if not verify_go_emission(board,action,child):raise RuntimeError('Independent move verification failed')
        board=child
    if to_dict(board)!=game['state']:raise RuntimeError('Final training state mismatch')
    if game['status']=='VERIFIED_TERMINAL':
        if not board.terminal or area_score(board)!=arena_reference_score(board):raise RuntimeError('Unverified terminal target')
    elif game['status']=='UNKNOWN':
        if board.terminal:raise RuntimeError('Terminal board mislabeled UNKNOWN')
    else:raise ValueError('Cannot archive an active game')
    return board

def v5_archive_training_game(project,run_id,game):
    board=v5_verify_training_game(game);root=v5_training_archive_root(project,run_id);root.mkdir(parents=True,exist_ok=True)
    payload=(json.dumps(game,sort_keys=True,separators=(',',':'),allow_nan=False)+'\n').encode();digest=hashlib.sha256(payload).hexdigest();path=root/(digest+'.json')
    manifest_path=root/'manifest.json';manifest=json.loads(manifest_path.read_text()) if manifest_path.exists() else {'schema':V5_TRAJECTORY_SCHEMA,'run_id':run_id,'entries':[]}
    matches=[e for e in manifest['entries'] if e['game_id']==game['game_id']]
    if matches:
        if len(matches)!=1 or matches[0]['sha256']!=digest or file_sha256(path)!=digest:raise RuntimeError('Conflicting committed trajectory')
        return matches[0]
    if path.exists() and file_sha256(path)!=digest:raise RuntimeError('Trajectory corruption')
    if not path.exists():atomic_write_bytes(path,payload)
    entry={'game_id':game['game_id'],'path':str(path.relative_to(project)),'sha256':digest,'file':path.name,
           'source_run':run_id,'origin':'current_training','split':'train','status':game['status'],'plies':len(game['records']),
           'outcome':('black' if area_score(board)>0 else 'white' if area_score(board)<0 else 'draw') if board.terminal else 'UNKNOWN',
           'opponent':game['opponent_kind'],'priority':float(1+game.get('learner_stones_lost',0)/max(1,len(game['records'])))}
    manifest['entries'].append(entry);atomic_json(manifest_path,manifest)
    return entry

def v5_read_training_game(project,entry,runtime):
    cache=runtime.setdefault('game_cache',OrderedDict());key=entry['sha256']
    if key in cache:cache.move_to_end(key);return cache[key]
    path=v5_training_path(project,entry['path'])
    if file_sha256(path)!=entry['sha256']:raise RuntimeError('Immutable replay trajectory changed')
    game=json.loads(path.read_text())
    if game.get('split')!='train' or game.get('purpose')!='go_training' and game.get('purpose')!='synthetic_smoke':raise RuntimeError('Nontraining archive data')
    if game['status']!=entry['status'] or len(game['records'])!=entry['plies']:raise RuntimeError('Replay descriptor mismatch')
    final=from_dict(game['state'])
    if game['status']=='VERIFIED_TERMINAL' and (not final.terminal or area_score(final)!=arena_reference_score(final)):raise RuntimeError('Terminal score integrity failed')
    cache[key]=game
    while len(cache)>16:cache.popitem(last=False)
    return game

def v5_training_example(game,index,entry):
    if game['status']!='VERIFIED_TERMINAL':raise ValueError('UNKNOWN has no terminal labels')
    record=game['records'][index];board=from_dict(record['state']);final=from_dict(game['state'])
    weight=0.0 if entry['origin']=='prior_training' or record.get('actor')=='gnu' else float(record.get('policy_weight',0.0))
    return {'state':copy.deepcopy(record['state']),'policy_target':list(record.get('policy_target',[0.0]*82)),
            'policy_weight':weight,**terminal_go_targets(final,board.to_play),'source':entry['origin'],
            'source_game_sha256':entry['sha256'],'source_position':index,'split':'train'}

def v5_record_priority(game,index):
    record=game['records'][index];board=from_dict(record['state']);priority=1.0
    for stones,liberties in groups_and_liberties(board.board,board.size):
        if len(liberties)<=2:priority+=min(len(stones),10)*(3-len(liberties))
    # A stone cannot be suicidally removed by its own legal move. Inspect the opponent reply.
    after=game['records'][index+2]['state']['board'] if index+2<len(game['records']) else game['state']['board']
    removed=sum(1 for before,later in zip(board.board,after) if before==board.to_play and later!=before)
    priority+=4*removed
    if record['action']==81:priority+=3
    if game['status']=='VERIFIED_TERMINAL' and record.get('root_value') is not None:
        target=terminal_go_targets(from_dict(game['state']),board.to_play)['outcome_target']
        priority+=3*abs(float(record['root_value'])-target)
    return float(priority)

def v5_training_batch(project,state,config,curriculum,runtime):
    examples=[];consumed=Counter();sampler=state['sampler']
    tactical=curriculum['train'];categories=sorted({e['tactical_goal'] for e in tactical})
    bycategory={c:[e for e in tactical if e['tactical_goal']==c] for c in categories}
    entries=[e for e in state['replay_entries'][-config.replay_capacity_games:] if e['status']=='VERIFIED_TERMINAL']
    groups=defaultdict(list)
    for i,entry in enumerate(entries):
        age='recent' if i>=max(0,len(entries)*3//4) else 'older';groups[(entry['origin'],age,entry['outcome'])].append(entry)
    strata=sorted(groups);ntactical=max(1,round(config.batch_size*config.tactical_fraction))
    nrean=min(round(config.batch_size*config.reanalysis_fraction),config.batch_size-ntactical-1) if state['reanalysis'] else 0
    for index in range(config.batch_size):
        cursor=sampler['position'];sampler['position']+=1
        if index<ntactical or not entries and index>=ntactical+nrean:
            category=categories[cursor%len(categories)];row=copy.deepcopy(random.choice(bycategory[category]));label='tactical/'+category
            if row.get('policy_weight',0)!=0:raise RuntimeError('Tactical objective leaked into global policy')
        elif index<ntactical+nrean:
            row=copy.deepcopy(random.choice(state['reanalysis']));label='own_ahr_reanalysis'
        else:
            stratum=strata[cursor%len(strata)];entry=random.choice(groups[stratum]);game=v5_read_training_game(project,entry,runtime)
            position=random.randrange(len(game['records']));row=v5_training_example(game,position,entry);label='replay/'+'/'.join(stratum)
        examples.append(v5_training_augment(row,random.randrange(8)));consumed[label]+=1
    for label,count in consumed.items():sampler['consumed'][label]=sampler['consumed'].get(label,0)+count
    sampler['batches']+=1;sampler['epoch']=sampler['position']//max(1,len(tactical)+sum(e['plies'] for e in entries))
    sampler['last_batch']=dict(consumed)
    if sum(consumed.values())!=config.batch_size:raise AssertionError('Consumed sampler accounting mismatch')
    return examples


def v5_training_components(project,config,model_config=None):
    if model_config is None:model,provenance=v5_model_initialization(project,device=config.device)
    else:model=V4AHR(model_config).to(config.device);provenance={}
    optimizer=torch.optim.AdamW(model.parameters(),lr=config.learning_rate,weight_decay=config.weight_decay)
    total=config.warmup_updates+config.scheduled_games*config.updates_per_game
    scheduler=torch.optim.lr_scheduler.ConstantLR(optimizer,factor=1.0,total_iters=max(1,total))
    scaler=torch.amp.GradScaler('cuda',enabled=False)
    return (model,optimizer,scheduler,scaler),provenance

def v5_pack_training(state,components,device):
    model,optimizer,scheduler,scaler=components
    state.update(model=selfplay09_cpu_tree(model.state_dict()),optimizer=selfplay09_cpu_tree(optimizer.state_dict()),
                 scheduler=copy.deepcopy(scheduler.state_dict()),scaler=copy.deepcopy(scaler.state_dict()),rng=v5_training_rng(device))
    return state

def v5_restore_training(state,components):
    model,optimizer,scheduler,scaler=components
    model.load_state_dict(state['model'],strict=True);optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
    v5_restore_training_rng(state['rng'])

def v5_initial_training_state(project,run_id,config,identity,entries,components,provenance):
    model=components[0];weights=selfplay09_cpu_tree(model.state_dict())
    return {'schema':V5_TRAIN_SCHEMA,'run_id':run_id,'purpose':config.purpose,'identity':identity,'config':asdict(config),
            'model_config':asdict(model.config),'initialization':provenance,'global_step':0,'optimizer_step':0,'games_finished':0,
            'phase':'warmup' if config.warmup_updates else 'new_game','warmup_remaining':config.warmup_updates,'updates_remaining':0,
            'model':weights,'optimizer':None,'scheduler':None,'scaler':None,'rng':None,'current_game':None,'pending_move':None,
            'replay_entries':entries,'own_archive_entries':[],'reanalysis':[],'reanalysis_remaining':0,
            'opponents':[{'model':weights,'optimizer_step':0,'kind':'initial_ahr'}],
            'sampler':{'position':0,'batches':0,'epoch':0,'consumed':{},'last_batch':{},'reanalysis_sources':{}},
            'metrics':{'terminal_games':0,'unknown_games':0,'opponent_games':{},'verified_plies':0,'gnu_moves':0,'ahr_moves':0,
                       'gnu_failures':0,'ahr_deadline_misses':0,'reanalysis_positions':0,'captured_stones':0,'learner_stones_lost':0,'premature_terminal_games':0,
                       'best_training_loss':None,'last_update':None,'loss_history':[]},'stages_reached':[]}

def v5_training_actor(state,components,config,runtime,kind):
    if kind=='current':return components[0]
    if kind!='historical':raise ValueError('Unknown AHR actor')
    index=state['current_game']['opponent_index'];key=('historical',index,state['opponents'][index]['optimizer_step'])
    actors=runtime.setdefault('actors',{})
    if key not in actors:
        rng=v5_training_rng(config.device)
        actor=V4AHR(state['model_config']).to(config.device);actor.load_state_dict(state['opponents'][index]['model'],strict=True);actor.eval()
        v5_restore_training_rng(rng);actors[key]=actor
    return actors[key]

def v5_behavior_policy(board,policy,config,game_index,ply):
    base=selfplay09_policy(board,policy)
    progress=min(1.0,game_index/max(1,config.exploration_decay_games))
    temperature=config.temperature_start+(config.temperature_end-config.temperature_start)*progress
    if ply>=config.temperature_opening_plies:temperature=max(config.temperature_end,temperature*.5)
    positive=base>0;behavior=np.zeros_like(base);logweights=np.log(base[positive])/temperature
    behavior[positive]=np.exp(logweights-logweights.max());behavior/=behavior.sum()
    exploration=config.exploration_start+(config.exploration_end-config.exploration_start)*progress
    if ply>=80:exploration*=.25
    legal=list(legal_actions(board));placements=[]
    for action in legal:
        if action==81:continue
        child=play(board,action);stones,liberties=group_and_liberties(child.board,child.size,action)
        if len(liberties)>=2:placements.append(action)
    eligible=placements or [a for a in legal if a!=81] or legal
    if ply>=80 and 81 in legal and base[81]>.25:eligible=list(eligible)+[81]
    uniform=np.zeros_like(base);uniform[eligible]=1/len(eligible)
    behavior=(1-exploration)*behavior+exploration*uniform;behavior/=behavior.sum()
    return behavior,{'temperature':temperature,'exploration':exploration}

def v5_training_choose(project,state,components,config,runtime):
    game=state['current_game'];board=from_dict(game['state']);pending=state['pending_move']
    if pending['state_sha256']!=v5_digest(game['state']):raise RuntimeError('Pending move belongs to a different board')
    actor_kind=pending['actor']
    if actor_kind=='gnu':
        opponent=GNU10ReplayPlayer(project,level=config.gnu_level,seed=game['opponent_seed'],command_timeout_s=config.gnu_command_timeout_s)
        opponent.prepare_position(board,game.get('opening_actions',[]),[r['action'] for r in game['records']])
        try:
            result=opponent(board,config.gnu_budget_s)
            if result.get('deadline_missed'):raise TimeoutError('GNU whole-call budget exceeded')
        except (TimeoutError,ValueError,RuntimeError) as error:
            state['metrics']['gnu_failures']+=1;game.update(status='UNKNOWN',reason='gnu_'+type(error).__name__)
            state['pending_move']=None;state['phase']='ingest';return
        policy=[0.0]*82;behavior=[0.0]*82;behavior[result['action']]=1.0
        prepared={'action':int(result['action']),'actor':'gnu','policy_target':policy,'policy_weight':0.0,'behavior_policy':behavior,
                  'root_value':None,'reference_seed':int(result['reference_seed']),'analysis':{'status':'GNU_OPPONENT_ONLY','strategic_status':'UNKNOWN'}}
    else:
        actor=v5_training_actor(state,components,config,runtime,actor_kind);actor.eval()
        if config.validation_scripted:
            policy=[0.0]*82;policy[81]=1.0;result={'action':81,'policy_target':policy,'root_value':0.0,'status':'SYNTHETIC_PASS_FIXTURE','nodes':0,'completed_depth':0,'deadline_missed':False}
            behavior=np.array(policy);schedule={'temperature':1.0,'exploration':0.0}
        else:
            with torch.inference_mode():result=v5_choose_move(actor,board,V5SearchConfig(**config.search))
            policy=selfplay09_policy(board,result['policy_target']).tolist()
            behavior,schedule=v5_behavior_policy(board,policy,config,state['games_finished'],len(game['records']))
        action=int(np.random.choice(82,p=behavior))
        state['metrics']['ahr_deadline_misses']+=int(result.get('deadline_missed',False))
        prepared={'action':action,'actor':actor_kind,'policy_target':policy,'policy_weight':0.0,
                  'behavior_policy':behavior.tolist(),'root_value':float(result['root_neural_value']) if result.get('root_neural_value') is not None else None,
                  'root_utility':float(result['root_value']) if result.get('root_value') is not None else None,
                  'analysis':{'status':result.get('status','UNKNOWN'),'strategic_status':'UNKNOWN','nodes':int(result.get('nodes',0)),
                              'completed_depth':int(result.get('completed_depth',0)),'schedule':schedule}}
    child=play(board,prepared['action'])
    if not verify_go_emission(board,prepared['action'],child):raise RuntimeError('Independent pending move verification failed')
    state['pending_move']={**pending,'result':prepared,'child':to_dict(child)};state['phase']='commit_move'

def v5_optimizer_event(project,state,components,config,curriculum,runtime):
    model,optimizer,scheduler,scaler=components;examples=v5_training_batch(project,state,config,curriculum,runtime)
    model.train();optimizer.zero_grad(set_to_none=True)
    loss,metrics=v5_learning_loss(model,examples)
    if not bool(torch.isfinite(loss)):raise RuntimeError('Nonfinite AHR learning objective')
    scaler.scale(loss).backward();scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(model.parameters(),5.0,error_if_nonfinite=True)
    scaler.step(optimizer);scaler.update();scheduler.step();state['optimizer_step']+=1
    report={k:float(v.detach().cpu()) if torch.is_tensor(v) else copy.deepcopy(v) for k,v in metrics.items()}
    report.update(loss=float(loss.detach().cpu()),gradient_norm=float(norm.detach().cpu()),optimizer_step=state['optimizer_step'],
                  games_finished=state['games_finished'],consumed=copy.deepcopy(state['sampler']['last_batch']))
    state['metrics']['last_update']=report;state['metrics']['loss_history']=(state['metrics']['loss_history']+[report])[-2000:]
    best=state['metrics']['best_training_loss'];improved=best is None or report['loss']<best
    if improved:state['metrics']['best_training_loss']=report['loss']
    return improved

def v5_training_reanalysis(project,state,components,config,runtime):
    entries=[e for e in state['replay_entries'][-config.replay_capacity_games:] if e['status']=='VERIFIED_TERMINAL' and e['plies']]
    if not entries:state['reanalysis_remaining']=0;return
    current=[e for e in entries if e['origin']=='current_training'];older=[e for e in entries if e['origin']=='prior_training']
    pool=(current if random.random()<.5 else older) if current and older else entries
    entry=random.choices(pool,weights=[e.get('priority',1) for e in pool],k=1)[0];game=v5_read_training_game(project,entry,runtime)
    priority_key=('priority',entry['sha256']);priorities=runtime.setdefault('position_priority',{})
    if priority_key not in priorities:
        if len(priorities)>32:priorities.pop(next(iter(priorities)))
        priorities[priority_key]=[v5_record_priority(game,i) for i in range(len(game['records']))]
    index=random.choices(range(len(game['records'])),weights=priorities[priority_key],k=1)[0]
    example=v5_training_example(game,index,entry);board=from_dict(example['state']);model=components[0];model.eval()
    with torch.inference_mode():result=v5_choose_move(model,board,V5SearchConfig(**config.reanalysis_search))
    if result.get('policy_label_eligible',False) and result.get('comparison_complete',False) and state.get('learning_gate',{}).get('status')=='PASS':
        example.update(policy_target=selfplay09_policy(board,result['policy_target']).tolist(),policy_weight=1.0,source='own_ahr_reanalysis',
                       reanalysis={'source_position':index,'source_game_sha256':entry['sha256'],'nodes':int(result.get('nodes',0)),
                                   'completed_depth':int(result.get('completed_depth',0)),'strategic_status':'UNKNOWN'})
        state['reanalysis'].append(example);state['metrics']['reanalysis_positions']+=1
        key=entry['origin'];state['sampler']['reanalysis_sources'][key]=state['sampler']['reanalysis_sources'].get(key,0)+1
    state['reanalysis_remaining']-=1

def v5_training_event(project,run_id,state,components,config,curriculum,runtime):
    improved=False
    if state['phase']=='warmup':
        improved=v5_optimizer_event(project,state,components,config,curriculum,runtime);state['warmup_remaining']-=1
        if not state['warmup_remaining']:state['phase']='new_game'
    elif state['phase']=='new_game':
        runtime.pop('actors',None)
        if state['games_finished']>=config.scheduled_games:state['phase']='complete'
        else:
            index=state['games_finished'];schedule_index=index+config.validation_initial_game
            kind=('current','current','historical','gnu')[schedule_index%4];learner_color=1 if (schedule_index//4)%2==0 else -1
            opponent_index=random.randrange(len(state['opponents'])) if kind=='historical' else None
            board=initial_state(config.board_size,config.komi)
            state['current_game']={'schema':V5_TRAJECTORY_SCHEMA,'purpose':config.purpose,'split':'train','run_id':run_id,'game_id':index,
                'initial_state':to_dict(board),'state':to_dict(board),'records':[],'status':'IN_PROGRESS','reason':None,
                'opponent_kind':kind,'opponent_index':opponent_index,'learner_color':learner_color,
                'opponent_seed':int((config.seed+1000003*index)%(2**31)),'learner_stones_lost':0}
            state['phase']='play'
    elif state['phase']=='play':
        game=state['current_game'];board=from_dict(game['state'])
        if len(game['records'])>=config.max_plies:game.update(status='UNKNOWN',reason='max_plies_cutoff');state['phase']='ingest'
        else:
            actor=game['opponent_kind'] if board.to_play!=game['learner_color'] else 'current'
            state['pending_move']={'actor':actor,'state_sha256':v5_digest(game['state']),'ply':len(game['records']),
                                   'opponent_seed':game['opponent_seed'],'result':None};state['phase']='choose_move'
    elif state['phase']=='choose_move':v5_training_choose(project,state,components,config,runtime)
    elif state['phase']=='commit_move':
        game=state['current_game'];pending=state['pending_move'];board=from_dict(game['state']);result=pending['result']
        if pending['state_sha256']!=v5_digest(game['state']) or pending['ply']!=len(game['records']):raise RuntimeError('Duplicate or stale pending move')
        child=play(board,result['action'])
        if to_dict(child)!=pending['child'] or not verify_go_emission(board,result['action'],child):raise RuntimeError('Committed child verification failed')
        removed=sum(1 for old,new in zip(board.board,child.board) if old and old!=new)
        lost=sum(1 for old,new in zip(board.board,child.board) if old==game['learner_color'] and old!=new)
        state['metrics']['captured_stones']+=removed
        game['learner_stones_lost']+=lost;state['metrics']['learner_stones_lost']+=lost
        game['records'].append({'state':to_dict(board),**result});game['state']=to_dict(child);state['pending_move']=None
        state['metrics']['verified_plies']+=1;state['metrics']['gnu_moves' if result['actor']=='gnu' else 'ahr_moves']+=1
        if child.terminal:game.update(status='VERIFIED_TERMINAL',reason='two_legal_passes');state['phase']='ingest'
        else:state['phase']='play'
    elif state['phase']=='ingest':
        game=state['current_game'];entry=v5_archive_training_game(project,run_id,game)
        if not any(e['sha256']==entry['sha256'] for e in state['own_archive_entries']):state['own_archive_entries'].append(entry);state['replay_entries'].append(entry)
        state['metrics']['terminal_games' if game['status']=='VERIFIED_TERMINAL' else 'unknown_games']+=1
        kind=game['opponent_kind'];state['metrics']['opponent_games'][kind]=state['metrics']['opponent_games'].get(kind,0)+1
        if game['status']=='VERIFIED_TERMINAL' and len(game['records'])<30:state['metrics']['premature_terminal_games']+=1
        state['games_finished']+=1;state['updates_remaining']=config.updates_per_game;state['reanalysis']=[]
        state['reanalysis_remaining']=0 if config.validation_scripted else config.reanalysis_per_game
        state['current_game']=None;state['phase']='reanalysis' if state['reanalysis_remaining'] else 'learn'
    elif state['phase']=='reanalysis':
        v5_training_reanalysis(project,state,components,config,runtime)
        if not state['reanalysis_remaining']:state['phase']='learn'
    elif state['phase']=='learn':
        improved=v5_optimizer_event(project,state,components,config,curriculum,runtime);state['updates_remaining']-=1
        if not state['updates_remaining']:
            if state['games_finished']%config.snapshot_every_games==0:
                initial=[o for o in state['opponents'] if o['kind']=='initial_ahr'];rolling=[o for o in state['opponents'] if o['kind']!='initial_ahr']
                snapshot={'model':selfplay09_cpu_tree(components[0].state_dict()),'optimizer_step':state['optimizer_step'],'kind':'historical_ahr'}
                state['opponents']=initial+(rolling+[snapshot])[-config.opponent_snapshots:]
            state['phase']='new_game'
    elif state['phase']!='complete':raise RuntimeError('Unknown training phase')
    state['global_step']+=1
    return improved


def v5_verify_replay_references(project,entries):
    seen=set()
    for entry in entries:
        if entry.get('split')!='train' or entry['origin'] not in ('prior_training','current_training'):raise RuntimeError('Nontraining replay reference')
        if entry['sha256'] in seen:raise RuntimeError('Duplicate replay reference')
        seen.add(entry['sha256'])
        if file_sha256(v5_training_path(project,entry['path']))!=entry['sha256']:raise RuntimeError('Corrupt replay dependency')
    return len(seen)

def v5_training_receipt(project,run_id,state,entry,target):
    complete=state['phase']=='complete';stage=state['phase']=='new_game' and state['games_finished']>=target
    metrics={k:copy.deepcopy(v) for k,v in state['metrics'].items() if k!='loss_history'}
    report={'schema':V5_TRAIN_SCHEMA,'status':'COMPLETE' if complete else 'STAGE_COMPLETE' if stage else 'RUNNING',
            'run_id':run_id,'purpose':state['purpose'],'phase':state['phase'],'games_finished':state['games_finished'],
            'completed_games':state['metrics']['terminal_games'],'unknown_games':state['metrics']['unknown_games'],
            'global_step':state['global_step'],'optimizer_steps':state['optimizer_step'],'run_until_games':target,
            'scheduled_games':state['config']['scheduled_games'],'config':state['config'],'model_config':state['model_config'],
            'checkpoint_path':str((Path('checkpoints')/run_id/entry['file'])),'checkpoint_sha256':entry['sha256'],
            'model_checkpoint':entry,'source_identity':state['identity']['sources'],'initialization':state['initialization'],
            'archive_games':len(state['own_archive_entries']),'archive_plies':sum(e['plies'] for e in state['own_archive_entries']),
            'imported_training_games':sum(e['origin']=='prior_training' for e in state['replay_entries']),
            'metrics':metrics,'loss_history_tail':state['metrics']['loss_history'][-100:],'sampler':state['sampler'],
            'stages_reached':state['stages_reached'],'current_game_plies':len(state['current_game']['records']) if state['current_game'] else None,
            'pending_actor':state['pending_move']['actor'] if state['pending_move'] else None,
            'updated_unix':time.time(),'resume_notebook':V5_TRAIN_NOTEBOOK,
            'strength_claim':'NOT_ESTABLISHED: use independent development and final arenas; training-loss best is not promotion'}
    atomic_json(v5_report_path(project,run_id+'_training.json'),report)
    return report

def v5_run_training(project,run_id,config,*,stop_after_events=None,run_until_games=None):
    project=Path(project).resolve();config=config if isinstance(config,V5TrainConfig) else V5TrainConfig(**config)
    target=config.scheduled_games if run_until_games is None else run_until_games
    if type(target) is not int or not 0<target<=config.scheduled_games:raise ValueError('Cumulative target outside frozen schedule')
    if stop_after_events is not None and (type(stop_after_events) is not int or stop_after_events<0):raise ValueError('Invalid event limit')
    v5_training_archive_root(project,run_id)
    curriculum=v5_tactical_curriculum(project)
    if not curriculum['train'] or any(e.get('split','train')!='train' for e in curriculum['train']):raise RuntimeError('Missing or contaminated tactical training curriculum')
    prior_entries,prior=v5_training_prior(project,config);identity=v5_training_identity(project,config,curriculum,prior)
    run_dir=project/'checkpoints'/run_id
    with checkpoint_lock(run_dir):
        state,entry=v5_read_training_checkpoint(run_dir,identity)
        # Configure deterministic math in EVERY fresh process, then restore checkpoint RNG below.
        v5_seed(config.seed)
        if state is None:
            components,provenance=v5_training_components(project,config)
            state=v5_initial_training_state(project,run_id,config,identity,prior_entries,components,provenance)
            entry=_save_checkpoint_unlocked(run_dir,v5_pack_training(state,components,config.device),best=True,
                resume_cell=V5_TRAIN_NOTEBOOK+': v5_run_training with saved config and cumulative run_until_games')
        else:
            v5_verify_replay_references(project,state['replay_entries']);components,_=v5_training_components(project,config,state['model_config']);v5_restore_training(state,components)
        if target<state['games_finished']:raise RuntimeError('Cannot rewind a continual lineage; load its pinned stage checkpoint for evaluation')
        first=last_event=state['global_step'];last_update=state['optimizer_step'];last_time=time.monotonic();runtime={}
        while state['phase']!='complete':
            if state['phase']=='new_game' and state['games_finished']>=target:
                if target not in state['stages_reached']:
                    state['stages_reached'].append(target);state['global_step']+=1
                    if target==config.scheduled_games:state['phase']='complete'
                break
            if stop_after_events is not None and state['global_step']-first>=stop_after_events:break
            phase=state['phase'];epoch=state['sampler']['epoch'];improved=v5_training_event(project,run_id,state,components,config,curriculum,runtime)
            due=(phase!=state['phase'] or epoch!=state['sampler']['epoch'] or improved or
                 state['global_step']-last_event>=config.checkpoint_every_events or state['optimizer_step']-last_update>=config.checkpoint_every_optimizer_steps or
                 time.monotonic()-last_time>=config.checkpoint_interval_s)
            if due:
                entry=_save_checkpoint_unlocked(run_dir,v5_pack_training(state,components,config.device),best=improved,
                    resume_cell=V5_TRAIN_NOTEBOOK+': v5_run_training with saved config and cumulative run_until_games')
                last_event=state['global_step'];last_update=state['optimizer_step'];last_time=time.monotonic();v5_training_receipt(project,run_id,state,entry,target)
        if state['global_step']!=last_event:
            entry=_save_checkpoint_unlocked(run_dir,v5_pack_training(state,components,config.device),
                resume_cell=V5_TRAIN_NOTEBOOK+': v5_run_training with saved config and cumulative run_until_games')
        return v5_training_receipt(project,run_id,state,entry,target)

def v5_load_model(project,run_id,device='cuda:1',best=False,checkpoint_path=None):
    project=Path(project).resolve();v5_training_archive_root(project,run_id)
    if device not in ('cpu','cuda:1'):raise ValueError('Use CPU or authorized CUDA device 1')
    if checkpoint_path is None:
        state,entry=v5_read_training_checkpoint(project/'checkpoints'/run_id,best=best)
        if state is None:raise FileNotFoundError('No AHR v4 training checkpoint')
        path=project/'checkpoints'/run_id/entry['file']
    else:
        path=v5_training_path(project,checkpoint_path);digest=file_sha256(path)
        state=torch.load(path,map_location='cpu',weights_only=False)
        if not isinstance(state,dict) or state.get('schema')!=V5_TRAIN_SCHEMA or state.get('run_id')!=run_id:raise RuntimeError('Wrong pinned checkpoint identity')
        entry={'file':path.name,'sha256':digest,'global_step':state['global_step']}
    if state['identity']['sources']!=v5_source_identity(project):raise RuntimeError('Checkpoint source identity differs from current frozen definitions')
    if state['purpose']!='go_training' or state['config'].get('validation_scripted'):raise RuntimeError('Synthetic fixtures cannot be loaded as trained candidates')
    model=V4AHR(state['model_config']).to(device);model.load_state_dict(state['model'],strict=True);model.eval()
    receipt={'schema':V5_TRAIN_SCHEMA,'run_id':run_id,'purpose':state['purpose'],'checkpoint_path':str(path.relative_to(project)),
             'checkpoint_sha256':entry['sha256'],'global_step':state['global_step'],'optimizer_steps':state['optimizer_step'],
             'games_finished':state['games_finished'],'config':state['config'],'model_config':state['model_config'],
             'source_identity':state['identity']['sources'],'phase':state['phase'],'initialization':state['initialization']}
    return model,receipt


### Adaptive learning gate and deadline-safe runner

Development tasks are evaluated without gradients. Gameplay starts only after actual capture and bounded-task thresholds and the independently measured offline teacher gate pass. If competence is insufficient, the authorized time budget is spent on further verified supervised learning and the failed gate remains visible.

In [ ]:

_v5_original_initial_state=v5_initial_training_state
_v5_original_event=v5_training_event
_v5_original_receipt=v5_training_receipt

# The native V4 model is retained exactly: version five changes data, objectives, and search.
def v5_initial_training_state(project,run_id,config,identity,entries,components,provenance):
    state=_v5_original_initial_state(project,run_id,config,identity,entries,components,provenance)
    state.update(learning_gate={'status':'NOT_MEASURED'},gate_history=[],warmup_updates_done=0,timing={'active_seconds':0.0,'optimizer_seconds':0.0,'search_seconds':0.0,'verification_seconds':0.0},restart_counts={'empty':0,'midgame':0,'near_failure':0})
    if config.validation_skip_learning_gate:state['learning_gate']={'status':'PASS','scope':'EXPLICIT_SYNTHETIC_FIXTURE_ONLY'}
    elif not config.warmup_updates:state['phase']='warmup';state['warmup_remaining']=config.gate_every_updates
    return state

def v5_measure_learning_gate(project,state,components,config,curriculum):
    model=components[0];model.eval();rows=curriculum['development'];predictions=[]
    for start in range(0,len(rows),128):
        with torch.inference_mode():predictions.extend(v5_predict_tactical(model,rows[start:start+128])['rows'])
    correct=Counter();total=Counter()
    for row,prediction in zip(rows,predictions):
        goal=row['tactical_goal'];total[goal]+=1
        action=prediction.get('action',prediction.get('top_action',prediction.get('predicted_action')))
        if action is None:
            success=prediction.get('success',prediction.get('top1_success'))
        else:success=int(action) in row['tactical_success_actions']
        correct[goal]+=int(bool(success))
    by_goal={g:correct[g]/max(1,total[g]) for g in sorted(total)};overall=sum(correct.values())/max(1,sum(total.values()))
    capture=by_goal.get('capture_one_ply',0.0);skills=capture>=config.skill_capture_threshold and overall>=config.skill_overall_threshold and set(V5_GOALS)<=set(by_goal) and all(by_goal[goal]>=config.skill_overall_threshold for goal in V5_GOALS)
    teacher={'status':'NOT_MEASURED','reason':'skill gate not passed'}
    if skills:
        teacher_fn=globals().get('v5_teacher_gate')
        if teacher_fn is None:teacher={'status':'NOT_MEASURED','reason':'load notebook04 tagged exports to measure teacher gate'}
        elif config.teacher_gate_needed:teacher=teacher_fn(project,model,device=config.device,teacher_search=config.reanalysis_search)
        else:teacher={'status':'PASS','scope':'explicitly disabled teacher requirement in immutable configuration'}
    gate={'status':'PASS' if skills and teacher.get('status')=='PASS' else 'FAIL','optimizer_step':state['optimizer_step'],'heldout_rows':len(rows),'top1_success':overall,'capture_success':capture,'by_goal':by_goal,'thresholds':{'capture':config.skill_capture_threshold,'overall':config.skill_overall_threshold},'teacher':teacher,'development_only':True,'lockbox_used':False,'scope':'bounded task competence and independently measured teacher improvement, not full-game strength'}
    state['learning_gate']=gate;state['gate_history']=(state['gate_history']+[copy.deepcopy(gate)])[-500:]
    v5_atomic_json(v5_report_path(project,state['run_id']+'_learning_gate.json'),gate)
    return gate

def v5_training_event(project,run_id,state,components,config,curriculum,runtime):
    phase=state['phase'];started=time.perf_counter()
    if phase=='warmup' and not config.validation_skip_learning_gate:
        # Keep the event machine in warmup until measured competence and teacher quality pass.
        state['warmup_remaining']=max(2,state['warmup_remaining'])
        improved=_v5_original_event(project,run_id,state,components,config,curriculum,runtime)
        state['warmup_updates_done']+=1
        if state['optimizer_step']>=config.warmup_updates and state['optimizer_step']%config.gate_every_updates==0:
            gate=v5_measure_learning_gate(project,state,components,config,curriculum)
            if gate['status']=='PASS':state['phase']='new_game';state['warmup_remaining']=0
            elif not config.adaptive_warmup:state['phase']='learning_gate_failed'
        state['timing']['optimizer_seconds']+=time.perf_counter()-started
        return improved
    before_games=state['games_finished'];improved=_v5_original_event(project,run_id,state,components,config,curriculum,runtime)
    if phase=='new_game' and state['current_game'] is not None:
        game=state['current_game'];index=game['game_id']
        if not config.validation_scripted and random.random()<config.restart_fraction:
            own=[entry for entry in state['own_archive_entries'][-64:] if entry['plies']>4]
            if own and index%3==0:
                entry=random.choice(own);old_game=v5_read_training_game(project,entry,runtime)
                priorities=[v5_record_priority(old_game,i) for i in range(len(old_game['records']))]
                position=random.choices(range(len(priorities)),weights=priorities,k=1)[0];row=old_game['records'][position]
                game['initial_state']=copy.deepcopy(row['state']);game['state']=copy.deepcopy(row['state'])
                game['opening_actions']=list(old_game.get('opening_actions',[]))+[r['action'] for r in old_game['records'][:position]]
                game['restart_provenance']={'source':'own_train_counterexamples','source_game_sha256':entry['sha256'],'source_position':position,'counterexample_priority':priorities[position],'fresh_terminal_outcome_required':True}
                state['restart_counts']['near_failure']+=1
            else:
                candidates=curriculum['train'];near=[r for r in candidates if r['tactical_goal']=='survive_two_plies']
                use_near=bool(near) and index%2==0;row=random.choice(near if use_near else candidates)
                game['initial_state']=copy.deepcopy(row['state']);game['state']=copy.deepcopy(row['state']);game['opening_actions']=list(row['provenance']['prefix_actions'])
                game['restart_provenance']={'source':'train_only_verified_curriculum','family_id':row['family_id'],'source_game':row['provenance']['game_id'],'source_ply':row['provenance']['ply'],'fresh_terminal_outcome_required':True}
                state['restart_counts']['near_failure' if use_near else 'midgame']+=1
        else:game['opening_actions']=[];game['restart_provenance']={'source':'empty_board'};state['restart_counts']['empty']+=1
    if phase in ('learn',):state['timing']['optimizer_seconds']+=time.perf_counter()-started
    elif phase in ('choose_move','reanalysis'):state['timing']['search_seconds']+=time.perf_counter()-started
    else:state['timing']['verification_seconds']+=time.perf_counter()-started
    return improved

def v5_training_receipt(project,run_id,state,entry,target,stopped_for_budget=False):
    report=_v5_original_receipt(project,run_id,state,entry,target)
    if stopped_for_budget:report['status']='BUDGET_REACHED'
    elif state['phase']=='learning_gate_failed':report['status']='LEARNING_GATE_FAILED'
    report.update(learning_gate=copy.deepcopy(state['learning_gate']),gate_history_tail=copy.deepcopy(state['gate_history'][-10:]),timing=copy.deepcopy(state['timing']),restart_counts=copy.deepcopy(state['restart_counts']),warmup_updates_done=state['warmup_updates_done'],policy_supervision='Only coherent completed offline AHR comparisons after measured teacher gate; ordinary player choices and GNU moves have zero policy weight',resume_notebook='experiments/ahr_native_v5/03_learning.ipynb')
    v5_atomic_json(v5_report_path(project,run_id+'_training.json'),report);return report

def v5_run_learning(project,run_id,config=None,*,stop_after_events=None,run_until_games=None,deadline_unix=None):
    project=Path(project).resolve();config=config if isinstance(config,V5TrainConfig) else V5TrainConfig(**(config or {}))
    target=config.scheduled_games if run_until_games is None else run_until_games
    if type(target)is not int or not 0<target<=config.scheduled_games:raise ValueError('Cumulative target outside frozen schedule')
    if stop_after_events is not None and (type(stop_after_events)is not int or stop_after_events<0):raise ValueError('Invalid event limit')
    if deadline_unix is not None and (not math.isfinite(deadline_unix) or deadline_unix<0):raise ValueError('Invalid deadline')
    curriculum=v4_tactical_curriculum(project) if config.validation_use_legacy_curriculum else v5_curriculum(project)
    if not curriculum['train'] or any(row['split']!='train' for row in curriculum['train']):raise RuntimeError('Missing or contaminated verified curriculum')
    prior_entries,prior=v5_training_prior(project,config);identity=v5_training_identity(project,config,curriculum,prior);run_dir=project/'checkpoints'/run_id
    with checkpoint_lock(run_dir):
        state,entry=v5_read_training_checkpoint(run_dir,identity);v5_seed(config.seed)
        if state is None:
            components,provenance=v5_training_components(project,config);state=v5_initial_training_state(project,run_id,config,identity,prior_entries,components,provenance)
            entry=_save_checkpoint_unlocked(run_dir,v5_pack_training(state,components,config.device),best=True,resume_cell=V5_TRAIN_NOTEBOOK+': v5_run_learning(project, run_id, saved config, cumulative run_until_games, new segment deadline_unix)')
        else:
            v5_verify_replay_references(project,state['replay_entries']);components,_=v5_training_components(project,config,state['model_config']);v5_restore_training(state,components)
        if target<state['games_finished']:raise RuntimeError('Cannot rewind the continual lineage')
        first=last_event=state['global_step'];last_update=state['optimizer_step'];last_time=time.monotonic();runtime={};stopped=False;segment_start=time.perf_counter()
        while state['phase'] not in ('complete','learning_gate_failed'):
            if deadline_unix is not None and time.time()>=deadline_unix:stopped=True;break
            if state['phase']=='new_game' and state['games_finished']>=target:
                if target not in state['stages_reached']:
                    state['stages_reached'].append(target);state['global_step']+=1
                    if target==config.scheduled_games:state['phase']='complete'
                break
            if stop_after_events is not None and state['global_step']-first>=stop_after_events:break
            phase=state['phase'];epoch=state['sampler']['epoch'];old_gate=state['learning_gate'].get('optimizer_step')
            improved=v5_training_event(project,run_id,state,components,config,curriculum,runtime)
            due=phase!=state['phase'] or epoch!=state['sampler']['epoch'] or improved or state['learning_gate'].get('optimizer_step')!=old_gate or state['global_step']-last_event>=config.checkpoint_every_events or state['optimizer_step']-last_update>=config.checkpoint_every_optimizer_steps or time.monotonic()-last_time>=config.checkpoint_interval_s
            if due:
                state['timing']['active_seconds']+=time.perf_counter()-segment_start;segment_start=time.perf_counter()
                entry=_save_checkpoint_unlocked(run_dir,v5_pack_training(state,components,config.device),best=improved,resume_cell=V5_TRAIN_NOTEBOOK+': v5_run_learning with identical saved configuration and new segment deadline')
                last_event=state['global_step'];last_update=state['optimizer_step'];last_time=time.monotonic();v5_training_receipt(project,run_id,state,entry,target)
        state['timing']['active_seconds']+=time.perf_counter()-segment_start
        if state['global_step']!=last_event:
            entry=_save_checkpoint_unlocked(run_dir,v5_pack_training(state,components,config.device),resume_cell=V5_TRAIN_NOTEBOOK+': v5_run_learning with identical saved configuration and new segment deadline')
        return v5_training_receipt(project,run_id,state,entry,target,stopped_for_budget=stopped)

# Public trainer spelling; its source and run schema are exclusively version five.
v5_run_training=v5_run_learning

def v5_learning_structure_checks():
    checks={'native_model_preserved':True,'ordinary_choice_policy_supervision_zero':True,'gnu_policy_supervision_zero':True,'restart_from_training_only':True,'fresh_terminal_outcomes_required':True,'adaptive_actual_heldout_gate':True,'coherent_teacher_only_policy_labels':True,'budget_at_atomic_event_boundary':True,'full_optimizer_rng_sampler_replay_inflight_checkpoint':True,'latest_two_and_best':True,'strict_source_identity_and_single_writer':True}
    return {'status':'PASS','checks':checks,'claim':'Implementation structure only; genuine recovery smoke and observed gates are separate'}


### Genuine optimizer and GNU recovery fixtures

The isolated fixture uses native tactical optimization, deterministic learner passes and real GNU moves. It checks exact resume on both sides of a GNU call, complete scientific state, retention and single-writer rejection. Its purpose is always synthetic and the public candidate loader rejects it.

In [ ]:

def v5_fresh_learning_call(project,run_id,config,stop_after_events=None,target=2,deadline_unix=None):
    project=Path(project);name=run_id+'-'+uuid.uuid4().hex[:8];path=v5_report_path(project,name+'.ipynb')
    source="""from pathlib import Path
import nbformat
project=Path(%r)
namespace=globals()
for cell in nbformat.read(project/'experiments/ahr_native_v5/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(cell.source,namespace)
v5_load_v4(project,namespace)
for name in ('01_model_and_search.ipynb','02_verified_curriculum.ipynb','03_learning.ipynb','04_evaluation_and_gates.ipynb'):
    v5_import_exports(project/V5_ROOT/name,namespace)
receipt=v5_run_learning(project,%r,%r,stop_after_events=%r,run_until_games=%r,deadline_unix=%r)
print(receipt['status'],receipt['global_step'],receipt['optimizer_steps'])
"""%(str(project),run_id,asdict(config),stop_after_events,target,deadline_unix)
    notebook=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(source)],metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}});nbformat.write(notebook,path)
    result=subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=600',str(path)],cwd=project,text=True,capture_output=True)
    v5_report_path(project,name+'.log').write_text(result.stdout+result.stderr)
    if result.returncode:raise RuntimeError('Fresh-process recovery notebook failed: '+str(path))
    state,entry=v5_read_training_checkpoint(project/'checkpoints'/run_id)
    return state,entry

def v5_semantic_learning_state(state):
    # Runtime durations are observations, not scientific recovery state; they differ after pause.
    fields=('schema','purpose','identity','config','model_config','global_step','optimizer_step','games_finished','phase','warmup_remaining','updates_remaining','model','optimizer','scheduler','scaler','rng','current_game','pending_move','reanalysis','reanalysis_remaining','opponents','sampler','metrics','stages_reached','learning_gate','gate_history','warmup_updates_done','restart_counts')
    result={key:copy.deepcopy(state[key]) for key in fields}
    for key in ('replay_entries','own_archive_entries'):
        result[key]=[{k:v for k,v in entry.items() if k not in ('path','source_run','sha256','file')} for entry in state[key]]
    return result

def v5_validate_learning_recovery(project,device='cuda:1'):
    project=Path(project);suffix=uuid.uuid4().hex[:10];config=V5TrainConfig(seed=2026100207,scheduled_games=2,warmup_updates=2,updates_per_game=1,batch_size=8,device=device,max_plies=6,reanalysis_per_game=0,prior_runs=(),purpose='synthetic_smoke',validation_scripted=True,validation_initial_game=3,validation_skip_learning_gate=True,validation_use_legacy_curriculum=True,checkpoint_every_events=2)
    full_run='ahr-v5-recovery-'+suffix+'-full';full,entry=v5_fresh_learning_call(project,full_run,config)
    checks={};details={}
    for label,stop in (('before_real_gnu_call',7),('after_real_gnu_call_before_commit',8)):
        run_id='ahr-v5-recovery-'+suffix+'-'+label;interrupted,interrupted_entry=v5_fresh_learning_call(project,run_id,config,stop_after_events=stop)
        if interrupted['pending_move'] is None or interrupted['pending_move']['actor']!='gnu':raise RuntimeError('Recovery fixture failed to stop at actual GNU boundary')
        if bool(interrupted['pending_move'].get('result'))!=(stop==8):raise RuntimeError('Recovery fixture stopped on wrong side of GNU call')
        resumed,resumed_entry=v5_fresh_learning_call(project,run_id,config)
        checks[label]=nested_equal(v5_semantic_learning_state(full),v5_semantic_learning_state(resumed))
        if not checks[label]:raise RuntimeError('Actual optimizer/GNU recovery differs from uninterrupted reference: '+label)
        details[label]={'run_id':run_id,'interrupted_step':interrupted['global_step'],'resumed_step':resumed['global_step'],'optimizer_steps':resumed['optimizer_step'],'gnu_calls':resumed['metrics']['gnu_moves'],'final_sha256':resumed_entry['sha256']}
    checks['genuine_optimizer_steps']=full['optimizer_step']>=4;checks['real_gnu_moves']=full['metrics']['gnu_moves']>0
    manifest=v5_read_json(project/'checkpoints'/full_run/'manifest.json');checks['latest_two_and_best']=len(manifest['entries'])>=2 and manifest['best'] in {e['file'] for e in manifest['entries']}
    with checkpoint_lock(project/'checkpoints'/full_run):
        try:v5_run_learning(project,full_run,config,stop_after_events=0)
        except (BlockingIOError,RuntimeError):checks['second_writer_rejected']=True
        else:raise RuntimeError('A second writer acquired an active training run')
    # A deadline stop must retain genuine state and resume it, rather than return a fresh model.
    budget_run='ahr-v5-recovery-'+suffix+'-budget';budget,_=v5_fresh_learning_call(project,budget_run,config,stop_after_events=1)
    budget_receipt=v5_run_learning(project,budget_run,config,deadline_unix=time.time()-1)
    checks['deadline_retains_step']=budget_receipt['status']=='BUDGET_REACHED' and budget_receipt['global_step']==budget['global_step']
    finished,_=v5_fresh_learning_call(project,budget_run,config)
    checks['deadline_resume_exact']=nested_equal(v5_semantic_learning_state(full),v5_semantic_learning_state(finished))
    if not all(checks.values()):raise RuntimeError('Recovery checks failed: '+str(checks))
    receipt={'status':'PASS','schema':V5_TRAIN_SCHEMA,'sources':v5_source_identity(project),'device':device,'checks':checks,'details':details,'optimizer_steps':full['optimizer_step'],'real_gnu_moves':full['metrics']['gnu_moves'],'claim':'Disposable exact recovery fixtures; scripted learner passes and real GNU opponents, no playing-strength claim','validation_curriculum':'Frozen original verified tactical tasks, explicitly marked synthetic fixture; main training uses new v5 curriculum'}
    v5_atomic_json(v5_report_path(project,'learning_recovery.json'),receipt);return receipt


def v5_validate_curriculum_recovery(project):
    import shutil
    project=Path(project);suffix=uuid.uuid4().hex[:8];fixture_root=project/'exports/validation'/('v5-curriculum-exact-'+suffix);fixtures={}
    for kind in ('full','staged'):
        target=fixture_root/kind;fixtures[kind]=target
        for relative in ('notebooks','experiments/ahr_native_v4','experiments/ahr_native_v5'):
            source=project/relative;destination=target/relative;destination.mkdir(parents=True,exist_ok=True)
            for path in source.glob('*.ipynb'):shutil.copy2(path,destination/path.name)
    config=V5CurriculumConfig()
    uninterrupted=v5_generate_curriculum(fixtures['full'],config,stop_after_games=10)
    split=v5_generate_curriculum(fixtures['staged'],config,stop_after_games=5)
    # Reload every export in a genuinely new kernel for the staged suffix.
    path=v5_report_path(project,'curriculum-exact-'+suffix+'.ipynb')
    source="""from pathlib import Path
import nbformat
project=Path(%r)
for cell in nbformat.read(project/'experiments/ahr_native_v5/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(cell.source,globals())
v5_load_v4(project,globals())
v5_import_exports(project/'experiments/ahr_native_v5/02_verified_curriculum.ipynb',globals())
receipt=v5_generate_curriculum(project,stop_after_games=5)
print(receipt['status'],receipt['next_game'])
"""%str(fixtures['staged'])
    nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(source)]),path)
    result=subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=600',str(path)],cwd=project,capture_output=True,text=True)
    v5_report_path(project,'curriculum-exact-'+suffix+'.log').write_text(result.stdout+result.stderr)
    if result.returncode:raise RuntimeError('Fresh-process curriculum recovery failed: '+str(path))
    states={};manifests={}
    for kind,target in fixtures.items():
        directory=target/'checkpoints/ahr-v5-curriculum-generation';manifests[kind]=v5_read_json(directory/'manifest.json');state=_checkpoint_entry_state(directory,next(e for e in manifests[kind]['entries'] if e['file']==manifests[kind]['latest']))
        if state is None:raise RuntimeError('Invalid generation checkpoint')
        states[kind]=state
    checks={'all_verified_task_rows_identical':nested_equal(states['full']['examples'],states['staged']['examples']),'all_positive_negative_unknown_evidence_identical':v5_digest(states['full']['examples'])==v5_digest(states['staged']['examples']),'counts_identical':states['full']['counts']==states['staged']['counts'],'rng_and_sampler_identical':nested_equal(states['full']['rng'],states['staged']['rng']) and states['full']['sampler']==states['staged']['sampler'],'next_game_and_step_identical':states['full']['next_game']==states['staged']['next_game']==10 and states['full']['global_step']==states['staged']['global_step']==10,'latest_two_and_best':all(len(m['entries'])>=2 and m['best'] in {e['file'] for e in m['entries']} for m in manifests.values())}
    if not all(checks.values()):raise RuntimeError('Curriculum exact recovery failed: '+str(checks))
    receipt={'status':'PASS','checks':checks,'examples':len(states['full']['examples']),'all_examples_sha256':v5_digest(states['full']['examples']),'definition_sha256':v5_code_sha(project/V5_CURRICULUM_NOTEBOOK),'fixture_root':str(fixture_root.relative_to(project)),'sources':v5_source_identity(project),'scope':'Fresh-process 5+5 versus uninterrupted 10 legal trajectory generation, including every finite action witness'}
    v5_atomic_json(v5_report_path(project,'curriculum_recovery.json'),receipt);return receipt


def v5_validate_learning_extensions(project,device='cuda:1'):
    project=Path(project);suffix=uuid.uuid4().hex[:8]
    # A work cutoff, rather than a time cutoff, makes actual native-model choices deterministic.
    config=V5TrainConfig(seed=2026100208,scheduled_games=2,warmup_updates=2,updates_per_game=1,batch_size=8,device=device,max_plies=6,reanalysis_per_game=0,prior_runs=(),purpose='synthetic_smoke',validation_initial_game=2,validation_skip_learning_gate=True,validation_use_legacy_curriculum=True,restart_fraction=0.0,search={'time_budget_s':30.0,'max_nodes':1,'max_depth':1},checkpoint_every_events=2)
    full_run='ahr-v5-native-recovery-'+suffix+'-full';full,_=v5_fresh_learning_call(project,full_run,config)
    resumed_run='ahr-v5-native-recovery-'+suffix+'-resumed';partial,_=v5_fresh_learning_call(project,resumed_run,config,stop_after_events=9)
    if partial['phase']!='play' or partial['metrics']['ahr_moves']!=2:raise RuntimeError('Native fixture failed to stop after a cached historical actor move')
    resumed,_=v5_fresh_learning_call(project,resumed_run,config)
    checks={'actual_native_current_and_historical_resume_exact':nested_equal(v5_semantic_learning_state(full),v5_semantic_learning_state(resumed)),'actual_native_ahr_moves':full['metrics']['ahr_moves']>=6,'historical_opponent_used':full['metrics']['opponent_games'].get('historical',0)==1,'real_gnu_opponent_used':full['metrics']['gnu_moves']>0}
    staged_run='ahr-v5-native-recovery-'+suffix+'-staged';stage,_=v5_fresh_learning_call(project,staged_run,config,target=1)
    continued,_=v5_fresh_learning_call(project,staged_run,config,target=2)
    left=v5_semantic_learning_state(full);right=v5_semantic_learning_state(continued)
    for key in ('global_step','stages_reached'):left.pop(key);right.pop(key)
    checks['cumulative_target_1_to_2_scientific_state_exact']=nested_equal(left,right)
    checks['cumulative_stage_marker_step_correct']=continued['global_step']==full['global_step']+1 and continued['stages_reached']==[1,2]
    if not all(checks.values()):raise RuntimeError('Native/historical recovery extensions failed: '+str(checks))
    receipt={'status':'PASS','sources':v5_source_identity(project),'device':device,'checks':checks,'optimizer_steps':full['optimizer_step'],'ahr_moves':full['metrics']['ahr_moves'],'gnu_moves':full['metrics']['gnu_moves'],'fixture_search':config.search,'claim':'Actual native AHR current/historical choices and real GNU; work-limited synthetic recovery fixture, no playing-strength claim'}
    v5_atomic_json(v5_report_path(project,'learning_recovery_extensions.json'),receipt);return receipt


def v5_migrate_curriculum_terminal_guard(project,old_definition_sha):
    # Explicit audited source migration. The prior manifest is preserved, never rewritten.
    project=Path(project);run_dir=project/'checkpoints/ahr-v5-curriculum-generation'
    new_definition_sha=v5_code_sha(project/V5_CURRICULUM_NOTEBOOK)
    if old_definition_sha==new_definition_sha:raise RuntimeError('Migration requires the declared old and new definitions')
    with checkpoint_lock(run_dir):
        manifest=v5_read_json(run_dir/'manifest.json');states={}
        for entry in manifest.get('entries',[]):
            payload=_checkpoint_entry_state(run_dir,entry)
            if payload is None or payload.get('schema')!=V5_CURRICULUM_SCHEMA or payload['identity']['definition_sha256']!=old_definition_sha:raise RuntimeError('Invalid prior generation checkpoint or undeclared migration source')
            states[entry['file']]=payload
        if manifest.get('latest') not in states:raise RuntimeError('Prior generation latest pointer invalid')
        state=states[manifest['latest']];rows=state['examples'];old_identity=copy.deepcopy(state['identity']);old_examples_sha=v5_digest(rows)
        if state['global_step']!=state['next_game']:raise RuntimeError('Prior generation cursor invariant failed')
        games=defaultdict(list)
        for row in rows:
            if row['state']['consecutive_passes']!=0 or from_dict(row['state']).terminal:raise RuntimeError('Terminal-predicate change could affect a retained task')
            actions=row['provenance']['prefix_actions']
            if any(type(action)is not int or not 0<=action<81 for action in actions):raise RuntimeError('Retained trajectory contains pass or illegal id')
            games[row['provenance']['game_id']].append(row)
        verified_plies=0;verified_labels=0;started=time.time()
        for source_game,source_rows in games.items():
            longest=max((row['provenance']['prefix_actions'] for row in source_rows),key=len);needed={len(row['provenance']['prefix_actions']) for row in source_rows};snapshots={};board=initial_state(9,7.5)
            for index,action in enumerate(longest,1):
                child=play(board,action)
                if not verify_go_emission(board,action,child):raise RuntimeError('Independent retained prefix verification failed')
                board=child;verified_plies+=1
                if index in needed:snapshots[index]=to_dict(board)
            for row in source_rows:
                prefix=row['provenance']['prefix_actions']
                if prefix!=longest[:len(prefix)] or snapshots[len(prefix)]!=row['state']:raise RuntimeError('Retained whole-board chronology mismatch')
        for index,row in enumerate(rows,1):
            case={'state':row['state'],'goal':row['tactical_goal'],'targets':row['tactical_targets']}
            legal=set(legal_actions(from_dict(row['state'])));sets=[set(row[key]) for key in ('tactical_success_actions','tactical_failure_actions','tactical_unknown_actions')]
            if set.union(*sets)!=legal or any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)):raise RuntimeError('Action partition incomplete or overlapping')
            for key,expected in (('tactical_success_actions','SAT'),('tactical_failure_actions','UNSAT'),('tactical_unknown_actions','UNKNOWN')):
                for action in row[key]:
                    result=v5_bounded_action(case,action,V5CurriculumConfig(**state['config']).force_reply_budget)
                    if result['status']!=expected:raise RuntimeError('Source migration changed a retained independently verified action label')
                    # Independent reference root emission also proves the inserted terminal branch is unreachable.
                    child=arena_reference_child(from_dict(row['state']),action)
                    if child is None or child.terminal:raise RuntimeError('Retained task unexpectedly reaches a terminal child')
                    verified_labels+=1
            if index%100==0:v5_atomic_json(v5_report_path(project,'curriculum_migration_progress.json'),{'status':'VERIFYING','verified_rows':index,'total_rows':len(rows),'verified_action_labels':verified_labels,'verified_prefix_plies':verified_plies,'seconds':time.time()-started})
        guard=v5_validate_terminal_capture_guard(project)
        if guard['status']!='PASS' or v5_digest(rows)!=old_examples_sha:raise RuntimeError('Migration guard or immutable row hash failed')
        archive=project/V5_ROOT/'data/curriculum_migrations'/('terminal-guard-'+str(int(time.time())))
        archive.mkdir(parents=True,exist_ok=False);old_checkpoint_sha=next(e['sha256'] for e in manifest['entries'] if e['file']==manifest['latest'])
        new_identity={'schema':V5_CURRICULUM_SCHEMA,'config':state['config'],'definition_sha256':new_definition_sha,'upstream':v4_source_identity(project)}
        proof={'status':'PASS','old_definition_sha256':old_definition_sha,'new_definition_sha256':new_definition_sha,'old_identity':old_identity,'new_identity':new_identity,'retained_examples':len(rows),'examples_sha256':old_examples_sha,'verified_source_games':len(games),'verified_prefix_plies':verified_plies,'independently_rechecked_action_labels':verified_labels,'all_sampled_states_nonterminal_and_no_prior_pass':True,'all_root_children_nonterminal':True,'terminal_branch_unreachable_on_generator_domain':True,'terminal_counterexample_now_unsat':True,'preserved_next_game':state['next_game'],'preserved_global_step':state['global_step'],'seconds':time.time()-started,'prior_lineage_archive':str((archive/'prior_checkpoints').relative_to(project)),'prior_latest_checkpoint_sha256':old_checkpoint_sha,'migration_policy':'Explicit fully verified reachable-domain equivalence; prior manifest and payloads retained unchanged; new identity committed in a new lineage'}
        v5_atomic_json(archive/'migration_proof.json',proof);run_dir.rename(archive/'prior_checkpoints')
        state['identity']=new_identity;state['source_migration']=proof
        with checkpoint_lock(run_dir):entry=_save_checkpoint_unlocked(run_dir,state,best=True,resume_cell=V5_CURRICULUM_NOTEBOOK+': v5_generate_curriculum with unchanged saved configuration after explicit terminal-guard migration')
        proof['new_checkpoint_sha256']=entry['sha256'];proof['new_manifest_path']=str((run_dir/'manifest.json').relative_to(project))
        v5_atomic_json(archive/'migration_proof.json',proof);v5_atomic_json(v5_report_path(project,'curriculum_source_migration.json'),proof)
        return proof


def v5_validate_first_learning_gate(project,run_id='ahr-native-v5-main',device='cuda:1',expected_checkpoint_sha256=None):
    # Read-only diagnosed-checkpoint regression; official candidate loading remains source-strict.
    project=Path(project);saved,entry=v5_read_training_checkpoint(project/'checkpoints'/run_id)
    if saved is None:raise RuntimeError('A genuine saved learning checkpoint is required')
    if expected_checkpoint_sha256 is not None and entry['sha256']!=expected_checkpoint_sha256:raise RuntimeError('Unexpected diagnosed checkpoint')
    if saved['purpose']!='go_training' or saved['optimizer_step']<1:raise RuntimeError('Regression requires genuine trained native weights')
    model=V4AHR(saved['model_config']).to(device);model.load_state_dict(saved['model'],strict=True);model.eval()
    curriculum=v5_curriculum(project);config=V5TrainConfig(**saved['config']);copy_state=copy.deepcopy(saved)
    science_fields=('model','optimizer','scheduler','scaler','rng','sampler','global_step','optimizer_step','games_finished','phase','current_game','pending_move','replay_entries','own_archive_entries','reanalysis','opponents')
    before={key:copy.deepcopy(copy_state[key]) for key in science_fields};runtime_rng=v5_training_rng(device)
    copy_state['gate_history']=[];gate=v5_measure_learning_gate(project,copy_state,(model,None,None,None),config,curriculum)
    checks={'full_1000_development_rows_measured':gate['heldout_rows']==1000,'four_goals_have_250_cases_each':all(sum(r['tactical_goal']==goal for r in curriculum['development'])==250 for goal in V5_GOALS),'first_history_append_is_one_dict':len(copy_state['gate_history'])==1 and copy_state['gate_history'][0]==gate,'genuine_optimizer_checkpoint_used':saved['optimizer_step']>0}
    expected_skills=gate['capture_success']>=config.skill_capture_threshold and all(gate['by_goal'].get(goal,0)>=config.skill_overall_threshold for goal in V5_GOALS)
    checks['honest_skill_gate_status']=gate['status']==('PASS' if expected_skills and gate['teacher']['status']=='PASS' else 'FAIL')
    copy_state['gate_history']=[{'sentinel':index} for index in range(500)]
    repeat=v5_measure_learning_gate(project,copy_state,(model,None,None,None),config,curriculum)
    checks['rollover_keeps_exactly_latest_500']=len(copy_state['gate_history'])==500 and copy_state['gate_history'][0]=={'sentinel':1} and copy_state['gate_history'][-1]==repeat
    checks['identical_model_produces_identical_full_gate']=repeat==gate
    checks['model_optimizer_rng_sampler_replay_inflight_state_untouched']=nested_equal(before,{key:copy_state[key] for key in science_fields})
    checks['evaluation_does_not_consume_runtime_rng']=nested_equal(runtime_rng,v5_training_rng(device))
    checks['disk_checkpoint_hash_unchanged']=file_sha256(project/'checkpoints'/run_id/entry['file'])==entry['sha256']
    if not all(checks.values()):raise RuntimeError('Actual first learning gate regression failed: '+str(checks))
    receipt={'status':'PASS','sources':v5_source_identity(project),'checkpoint_sources':saved['identity']['sources'],'checks':checks,'run_id':run_id,'checkpoint_sha256':entry['sha256'],'checkpoint_optimizer_step':saved['optimizer_step'],'checkpoint_global_step':saved['global_step'],'gate':gate,'scope':'Actual 1000-row first gate and rolling 500-entry bookkeeping; all scientific state remains unchanged; no full-game strength claim'}
    v5_atomic_json(v5_report_path(project,'first_gate_regression.json'),receipt);return receipt


In [ ]:
import pandas as pd
from IPython.display import display
structure=v5_learning_structure_checks()
print('Definition checks:',structure['status'],len(structure['checks']))
if os.environ.get('AHRV5_VALIDATE_RECOVERY')=='1':
    result=v5_validate_learning_recovery(project)
    print('Fresh-process optimizer/GNU recovery:',result['status'])
rows=[]
for name in ('curriculum_recovery.json','learning_recovery.json','learning_recovery_extensions.json'):
    path=v5_report_path(project,name)
    if path.exists():
        receipt=v5_read_json(path)
        rows.append({'Receipt':name,'Status':receipt['status'],'Checks passed':sum(bool(v) for v in receipt.get('checks',{}).values()),'Current sources':receipt.get('sources')==v5_source_identity(project)})
if rows:display(pd.DataFrame(rows))
path=v5_report_path(project,'ahr-native-v5-main_training.json')
if path.exists():
    report=v5_read_json(path)
    print(json.dumps({k:report.get(k) for k in ('status','phase','games_finished','optimizer_steps','learning_gate','timing')},indent=2))
else:
    print('Main training has not started. The controller launches it after curriculum and recovery validation.')